# Post 3 Scratch — Feature Upgrade: Var_v and Factor Analysis

**Goal**: Validate Var_v as the value-side complement to KL, then build a first-pass circuit graph via factor analysis on log(Var_v).

## Experiments
1. Compute Var_v for all heads on the Post 2 IOI / non-IOI prompt sets
2. Test ΔVar_v discrimination: circuit vs non-circuit (Mann-Whitney U)
3. Check corr(ΔKL, ΔVar_v) — should be << corr(ΔKL, Δχ) ≈ 0.70
4. Histogram of log(Var_v) — check approximate normality
5. Factor analysis on log(Var_v): fit W_act, W_null via sklearn FactorAnalysis; compute J_diff = W_act Wᵀ_act − W_null Wᵀ_null
6. Visualize J_diff as a circuit graph over heads

**Key claim**: KL is query-key side; Var_v is value-side → genuinely independent axes.  
**Prediction**: corr(ΔKL, ΔVar_v) << corr(ΔKL, Δχ) ≈ 0.70

---
*Scratch notebook — messy, do not publish.*

In [ ]:
import os
os.makedirs("figs", exist_ok=True)

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)

model = HookedTransformer.from_pretrained("gpt2-small")
print(f"Loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads/layer, d_head={model.cfg.d_head}")

## IOI Circuit Head Labels
From Wang et al. 2022 (arXiv:2211.00593) — same as Post 2.

In [ ]:
IOI_HEADS = {
    "Name Mover":          [(9, 6), (9, 9), (10, 0)],
    "Backup Name Mover":   [(9, 0), (9, 7), (10, 1), (10, 2), (10, 6), (10, 10), (11, 2), (11, 9)],
    "Negative Name Mover": [(10, 7), (11, 10)],
    "S-Inhibition":        [(7, 3), (7, 9), (8, 6), (8, 10)],
    "Induction":           [(5, 5), (6, 9)],
    "Duplicate Token":     [(0, 1), (3, 0)],
    "Previous Token":      [(2, 2), (4, 11)],
}

ALL_CIRCUIT_HEADS = set(lh for hs in IOI_HEADS.values() for lh in hs)
HL_FLAT = [(l, h) for l in range(12) for h in range(12)]  # 144 heads

def head_role(layer, head):
    for role, heads in IOI_HEADS.items():
        if (layer, head) in heads:
            return role
    return "Non-circuit"

print(f"Total circuit heads: {len(ALL_CIRCUIT_HEADS)}")
for role, heads in IOI_HEADS.items():
    print(f"  {role}: {heads}")

## Prompt Generation
Same seeds and logic as Post 2 — results should be identical to the Post 2 KL/χ values.

In [ ]:
import random

TEMPLATES_ABBA = [
    "When {A} and {B} went to the store, {B} gave a drink to",
    "When {A} and {B} went to the park, {B} handed a ball to",
    "When {A} and {B} arrived at the office, {B} passed a note to",
    "When {A} and {B} got to the restaurant, {B} offered a menu to",
    "When {A} and {B} walked into the room, {B} showed a book to",
    "After {A} and {B} met at the cafe, {B} sent a message to",
    "After {A} and {B} sat down for dinner, {B} gave a gift to",
]

TEMPLATES_BABA = [
    "When {B} and {A} went to the store, {B} gave a drink to",
    "When {B} and {A} went to the park, {B} handed a ball to",
    "When {B} and {A} arrived at the office, {B} passed a note to",
    "When {B} and {A} got to the restaurant, {B} offered a menu to",
    "When {B} and {A} walked into the room, {B} showed a book to",
    "After {B} and {A} met at the cafe, {B} sent a message to",
    "After {B} and {A} sat down for dinner, {B} gave a gift to",
]

# Non-IOI: third name C, no repetition → IOI circuit shouldn't fire.
TEMPLATES_NON_IOI = [
    "When {A} and {B} went to the store, {C} gave a drink to",
    "When {A} and {B} went to the park, {C} handed a ball to",
    "When {A} and {B} arrived at the office, {C} passed a note to",
    "When {A} and {B} got to the restaurant, {C} offered a menu to",
    "When {A} and {B} walked into the room, {C} showed a book to",
    "After {A} and {B} met at the cafe, {C} sent a message to",
    "After {A} and {B} sat down for dinner, {C} gave a gift to",
]

NAMES = [
    "Mary", "John", "Alice", "Bob", "Sarah", "Tom",
    "Emma", "James", "Lisa", "David", "Kate", "Mark",
]

def generate_unique_prompts_ioi(n, seed=42):
    random.seed(seed)
    templates = TEMPLATES_ABBA + TEMPLATES_BABA
    seen, prompts = set(), []
    while len(prompts) < n:
        template = random.choice(templates)
        a, b = random.sample(NAMES, 2)
        prompt = template.format(A=a, B=b)
        if prompt not in seen:
            seen.add(prompt)
            prompts.append(prompt)
    return prompts

def generate_unique_prompts_non_ioi(n, seed=43):
    random.seed(seed)
    seen, prompts = set(), []
    while len(prompts) < n:
        template = random.choice(TEMPLATES_NON_IOI)
        a, b, c = random.sample(NAMES, 3)
        prompt = template.format(A=a, B=b, C=c)
        if prompt not in seen:
            seen.add(prompt)
            prompts.append(prompt)
    return prompts

# 1000 prompts — first 50 identical to Post 2 (same seeds, same generation order)
ioi_prompts     = generate_unique_prompts_ioi(1000)
non_ioi_prompts = generate_unique_prompts_non_ioi(1000)
print(f"IOI: {len(ioi_prompts)}, Non-IOI: {len(non_ioi_prompts)}")
print(f"  IOI[0]: {ioi_prompts[0]}")
print(f"  Non-IOI[0]: {non_ioi_prompts[0]}")


## Diagnostic Computation: KL, χ, and Var_v

**KL** and **χ** are query-key side: they depend only on the attention pattern π = softmax(QKᵀ/√d_k).  
**Var_v** is value-side: it depends on the value vectors v = X W_V.

$$\text{Var}_v = \frac{1}{d_{\text{head}}} \text{Tr}[\text{Cov}_\pi(v)]$$

where the attention-weighted covariance is:
$$\text{Cov}_\pi(v) = \sum_i \pi_i v_i v_i^\top - \left(\sum_i \pi_i v_i\right)\left(\sum_i \pi_i v_i\right)^\top$$

So per head per prompt:
$$\text{Var}_v = \frac{1}{d_{\text{head}}} \left[ \sum_i \pi_i \|v_i\|^2 - \left\|\sum_i \pi_i v_i\right\|^2 \right]$$

This is the mean variance of the value vectors under the attention distribution.  
High Var_v → attention spreads over diverse value vectors.  
Low Var_v → all attended values are similar (either uniform over similar values, or concentrated on one).

**All computed at the last token position** (where IOI prediction is made).

In [ ]:
def compute_all_diagnostics(model, prompts):
    """
    Compute KL, chi (Var of log-attention), and Var_v for all heads at last token.

    Returns:
        kl:    (n_prompts, n_layers, n_heads)  — normalized KL
        chi:   (n_prompts, n_layers, n_heads)  — normalized Var_pi(log pi)
        varv:  (n_prompts, n_layers, n_heads)  — Var_v (NOT normalized; log-transform later)
    """
    n_layers = model.cfg.n_layers
    n_heads  = model.cfg.n_heads
    d_head   = model.cfg.d_head
    n_prompts = len(prompts)

    kl_all   = np.zeros((n_prompts, n_layers, n_heads))
    chi_all  = np.zeros((n_prompts, n_layers, n_heads))
    varv_all = np.zeros((n_prompts, n_layers, n_heads))

    for p_idx, prompt in enumerate(prompts):
        tokens = model.to_tokens(prompt)          # (1, seq_len)
        _, cache = model.run_with_cache(tokens)

        seq_len = tokens.shape[1]
        log_n   = np.log(seq_len)

        for layer in range(n_layers):
            # -- attention pattern at last token --
            # cache["pattern", layer]: (1, n_heads, dest, src)
            attn = cache["pattern", layer]          # (1, n_heads, seq_len, seq_len)
            pi   = attn[0, :, -1, :]                # (n_heads, seq_len) — from last token

            log_pi = torch.log(pi + 1e-12)          # (n_heads, seq_len)

            # KL(pi || u) = log n - H(pi), normalized
            H = -(pi * log_pi).sum(dim=-1)          # (n_heads,)
            kl_all[p_idx, layer, :] = (log_n - H.cpu().numpy()) / log_n

            # chi = Var_pi(log pi), normalized
            mean_log_pi = (pi * log_pi).sum(dim=-1, keepdim=True)  # (n_heads, 1)
            var_log_pi  = (pi * (log_pi - mean_log_pi) ** 2).sum(dim=-1)
            chi_all[p_idx, layer, :] = var_log_pi.cpu().numpy() / (log_n ** 2)

            # -- value vectors at all positions --
            # cache["v", layer]: (1, seq_len, n_heads, d_head)
            v = cache["v", layer][0, :, :, :]      # (seq_len, n_heads, d_head)
            v = v.permute(1, 0, 2)                  # (n_heads, seq_len, d_head)

            # Var_v = (1/d_head) * Tr[Cov_pi(v)]
            #       = (1/d_head) * [sum_i pi_i ||v_i||^2 - ||sum_i pi_i v_i||^2]
            pi_e = pi.unsqueeze(-1)                 # (n_heads, seq_len, 1) — for broadcasting
            # mean value vector under pi: (n_heads, d_head)
            mu_v = (pi_e * v).sum(dim=1)            # (n_heads, d_head)
            # E_pi[||v||^2] — scalar per head
            E_v2 = (pi * (v ** 2).sum(dim=-1)).sum(dim=-1)  # (n_heads,)
            # ||mu_v||^2 — scalar per head
            mu_v_sq = (mu_v ** 2).sum(dim=-1)       # (n_heads,)

            varv_raw = (E_v2 - mu_v_sq) / d_head    # (n_heads,)
            varv_all[p_idx, layer, :] = varv_raw.cpu().numpy()

    return kl_all, chi_all, varv_all


print("Computing diagnostics on IOI prompts...")
kl_ioi, chi_ioi, varv_ioi = compute_all_diagnostics(model, ioi_prompts)
print(f"  Shape: {kl_ioi.shape}")

print("Computing diagnostics on non-IOI prompts...")
kl_non, chi_non, varv_non = compute_all_diagnostics(model, non_ioi_prompts)
print(f"  Shape: {kl_non.shape}")

print("Done!")
print(f"\nVar_v range (IOI):     [{varv_ioi.min():.4f}, {varv_ioi.max():.4f}]")
print(f"Var_v range (non-IOI): [{varv_non.min():.4f}, {varv_non.max():.4f}]")

## Experiment 1 — ΔVar_v: Circuit vs Non-circuit

Does |ΔVar_v| (IOI − non-IOI) discriminate circuit vs non-circuit heads?  
Hypothesis: yes, Mann-Whitney p << 0.05.  
Also report mean ΔVar_v by circuit role to see which roles drive the signal.

In [ ]:
# Deltas: mean over prompts
delta_kl   = kl_ioi.mean(axis=0)   - kl_non.mean(axis=0)   # (12,12)
delta_chi  = chi_ioi.mean(axis=0)  - chi_non.mean(axis=0)
delta_varv = varv_ioi.mean(axis=0) - varv_non.mean(axis=0)

# Split into circuit vs non-circuit
abs_delta = {
    "kl":   {"circuit": [], "non-circuit": []},
    "chi":  {"circuit": [], "non-circuit": []},
    "varv": {"circuit": [], "non-circuit": []},
}

for l in range(12):
    for h in range(12):
        tag = "circuit" if (l, h) in ALL_CIRCUIT_HEADS else "non-circuit"
        abs_delta["kl"][tag].append(abs(delta_kl[l, h]))
        abs_delta["chi"][tag].append(abs(delta_chi[l, h]))
        abs_delta["varv"][tag].append(abs(delta_varv[l, h]))

for key in ["kl", "chi", "varv"]:
    circ = np.array(abs_delta[key]["circuit"])
    nonc = np.array(abs_delta[key]["non-circuit"])
    u, p = stats.mannwhitneyu(circ, nonc, alternative="greater")
    n_c, n_nc = len(circ), len(nonc)
    print(f"|Δ{key}|  circuit: {circ.mean():.4f} ± {circ.std():.4f}  "
          f"non-circuit: {nonc.mean():.4f} ± {nonc.std():.4f}  "
          f"MW p={p:.5f}")

In [ ]:
# Histogram: |Δvarv| circuit vs non-circuit
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, label in zip(axes, ["kl", "chi", "varv"], ["KL", "χ", "Var_v"]):
    circ = np.array(abs_delta[key]["circuit"])
    nonc = np.array(abs_delta[key]["non-circuit"])
    _, p = stats.mannwhitneyu(circ, nonc, alternative="greater")
    ax.hist(nonc, bins=20, alpha=0.5, color="#cccccc", label="Non-circuit", density=True)
    ax.hist(circ, bins=12, alpha=0.7, color="#d62728", label="Circuit", density=True)
    ax.set_xlabel(f"|Δ{label}|", fontsize=12)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"|Δ{label}| distribution\np = {p:.5f}", fontsize=12)
    ax.legend()

plt.tight_layout()
plt.savefig("figs/exp1_delta_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## Experiment 2 — corr(ΔKL, ΔVar_v) vs corr(ΔKL, Δχ)

Key test of the KL/Var_v independence claim.  
**Prediction**: corr(ΔKL, ΔVar_v) << corr(ΔKL, Δχ) ≈ 0.70  
If true: KL and Var_v carry independent information → worth including both.  
If false (high corr): Var_v may still be preferable as the richer feature, but the justification changes.

In [ ]:
dkl_flat   = delta_kl.flatten()
dchi_flat  = delta_chi.flatten()
dvarv_flat = delta_varv.flatten()

r_kl_chi,  p_kl_chi  = stats.pearsonr(dkl_flat, dchi_flat)
r_kl_varv, p_kl_varv = stats.pearsonr(dkl_flat, dvarv_flat)

print(f"corr(ΔKL, Δχ):    r = {r_kl_chi:.3f}  (p = {p_kl_chi:.4e})")
print(f"corr(ΔKL, ΔVar_v): r = {r_kl_varv:.3f}  (p = {p_kl_varv:.4e})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, x, y, xlabel, ylabel, r in [
    (axes[0], dkl_flat, dchi_flat,  "ΔKL", "Δχ",     r_kl_chi),
    (axes[1], dkl_flat, dvarv_flat, "ΔKL", "ΔVar_v", r_kl_varv),
]:
    colors = ["#d62728" if lh in ALL_CIRCUIT_HEADS else "#aaaaaa"
              for lh in HL_FLAT]
    ax.scatter(x, y, c=colors, s=20, alpha=0.7)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"r = {r:.3f}", fontsize=13)
    ax.axhline(0, color="k", lw=0.5, alpha=0.3)
    ax.axvline(0, color="k", lw=0.5, alpha=0.3)

axes[0].set_title(f"ΔKL vs Δχ\nr = {r_kl_chi:.3f}  (expected ≈ 0.70)", fontsize=12)
axes[1].set_title(f"ΔKL vs ΔVar_v\nr = {r_kl_varv:.3f}  (expected << 0.70)", fontsize=12)

plt.suptitle("Correlation of deltas across all 144 heads (red = circuit)", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("figs/exp2_delta_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## Experiment 3 — log(Var_v) Histogram

Check approximate normality of log(Var_v) across prompts for a few representative heads.  
Rationale for the log transform: Var_v ≥ 0, spans orders of magnitude → log maps to ℝ, compresses outliers, and ratios more natural than differences for multiplicative quantities.

Note: We do NOT expect perfect normality — the point is that the Gaussian factor model is a reasonable first approximation.

In [ ]:
# Global floor to avoid log(0)
EPS = 1e-8
log_varv_ioi = np.log(varv_ioi + EPS)  # (50, 12, 12)
log_varv_non = np.log(varv_non + EPS)

# Show histograms for a selection of circuit heads and non-circuit heads
SHOW_HEADS = {
    "Name Mover": [(9, 9), (9, 6), (10, 0)],
    "S-Inhibition": [(7, 3), (7, 9)],
    "Non-circuit (example)": [(0, 0), (1, 1), (6, 6)],
}

all_heads_to_show = [lh for heads in SHOW_HEADS.values() for lh in heads]
n_show = len(all_heads_to_show)
ncols = 4
nrows = (n_show + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes_flat = axes.flatten()

for idx, (l, h) in enumerate(all_heads_to_show):
    ax = axes_flat[idx]
    vals_ioi = log_varv_ioi[:, l, h]
    vals_non = log_varv_non[:, l, h]
    role = head_role(l, h)
    color = "#d62728" if (l, h) in ALL_CIRCUIT_HEADS else "#aaaaaa"
    ax.hist(vals_non, bins=15, alpha=0.4, color="#cccccc", label="non-IOI", density=True)
    ax.hist(vals_ioi, bins=15, alpha=0.6, color=color, label="IOI", density=True)

    # Overlay Gaussian fit
    all_vals = np.concatenate([vals_ioi, vals_non])
    mu, sigma = all_vals.mean(), all_vals.std()
    xs = np.linspace(all_vals.min(), all_vals.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, mu, sigma), 'k--', lw=1.2, alpha=0.7, label="N(μ,σ²)")

    ax.set_title(f"L{l}H{h} ({role[:8]})", fontsize=10)
    ax.set_xlabel("log(Var_v)", fontsize=9)
    ax.tick_params(labelsize=8)
    if idx == 0:
        ax.legend(fontsize=7)

for idx in range(n_show, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle("Histograms of log(Var_v): IOI (colored) vs non-IOI (gray) + Gaussian fit",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("figs/exp3_log_varv_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Quantitative normality check: Shapiro-Wilk on log(Var_v) per head, pooled IOI+non-IOI
sw_stats = []
for l in range(12):
    for h in range(12):
        vals = np.concatenate([log_varv_ioi[:, l, h], log_varv_non[:, l, h]])
        w, p_sw = stats.shapiro(vals)
        sw_stats.append((l, h, w, p_sw))

sw_arr = np.array([s[2] for s in sw_stats])
p_sw_arr = np.array([s[3] for s in sw_stats])
print(f"Shapiro-Wilk W across 144 heads: mean={sw_arr.mean():.3f}, min={sw_arr.min():.3f}")
print(f"Fraction with p > 0.05 (not rejected): {(p_sw_arr > 0.05).mean():.2f}")
print(f"(High fraction good — most heads' log(Var_v) are approximately normal)")

## Experiment 4 — (KL, Var_v) 2D Phase Space

Mirror of Post 2's (KL, χ) fingerprint plots, but with Var_v on the y-axis.  
Do circuit heads show clearer separation in (KL, Var_v) vs (KL, χ)?

In [ ]:
# Show a single name mover head in both (KL, chi) and (KL, Var_v)
l, h = 9, 9  # Name Mover

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_ioi, y_non, ylabel in [
    (axes[0], chi_ioi[:, l, h],   chi_non[:, l, h],   r"Norm. $\chi$"),
    (axes[1], varv_ioi[:, l, h],  varv_non[:, l, h],  r"$\mathrm{Var}_v$"),
]:
    ax.scatter(kl_non[:, l, h],  y_non,  alpha=0.4, s=30, c="#aaaaaa", label="Non-IOI", zorder=2)
    ax.scatter(kl_ioi[:, l, h],  y_ioi,  alpha=0.4, s=30, c="#d62728", label="IOI", zorder=2)

    m_act   = (kl_ioi[:, l, h].mean(),  y_ioi.mean())
    m_inact = (kl_non[:, l, h].mean(),  y_non.mean())
    ax.scatter(*m_act,   s=200, c="#d62728", marker="*", edgecolor="k", zorder=5)
    ax.scatter(*m_inact, s=200, c="#aaaaaa", marker="*", edgecolor="k", zorder=5)
    ax.annotate("", xy=m_act, xytext=m_inact,
                arrowprops=dict(arrowstyle="->", color="black", lw=1.5))

    ax.set_xlabel(r"Norm. KL: $\hat{\rho}_{\mathrm{eff}} / \log n$", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.legend(fontsize=10, frameon=False)
    ax.set_title(f"L{l}H{h} (Name Mover)", fontsize=13)

plt.suptitle("(KL, χ) vs (KL, Var_v) phase space — same head", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("figs/exp4_kl_varv_vs_kl_chi.png", dpi=150, bbox_inches="tight")
plt.show()

## Factor Analysis Setup

Fit a Gaussian factor model on log(Var_v) separately on IOI and non-IOI prompts:

$$\log(\text{Var}_v) = W h + \varepsilon, \quad h \sim \mathcal{N}(0, I), \quad \varepsilon \sim \mathcal{N}(0, \Lambda)$$

Then compute:
$$J_{\text{diff}} = W_{\text{act}} W_{\text{act}}^\top - W_{\text{null}} W_{\text{null}}^\top$$

This is the circuit-specific coupling matrix — shared structure (layer-depth confound) cancels.

Feature vector: log(Var_v) at last token, all 144 heads. (n_prompts=50, n_features=144)

In [ ]:
from sklearn.decomposition import FactorAnalysis

# Feature matrices: (n_prompts, 144)
X_act  = log_varv_ioi.reshape(50, -1)   # IOI activating
X_null = log_varv_non.reshape(50, -1)   # non-IOI null

print(f"X_act shape:  {X_act.shape}")
print(f"X_null shape: {X_null.shape}")

# Standardize each feature (head) to zero mean — important before FA
# NOTE: standardize separately for each prompt set (they may have different baselines)
X_act_c  = X_act  - X_act.mean(axis=0)
X_null_c = X_null - X_null.mean(axis=0)

print(f"\nX_act  mean after centering: {X_act_c.mean():.2e}")
print(f"X_null mean after centering: {X_null_c.mean():.2e}")

In [ ]:
# Fit factor analysis — sweep over n_components to find elbow
# n=50 prompts, p=144 features → factor model is underdetermined for large k
# Limit to k <= n_prompts // 3 ~ 16 to stay stable

K_RANGE = list(range(1, 17))
loglik_act  = []
loglik_null = []

for k in K_RANGE:
    fa_act  = FactorAnalysis(n_components=k, random_state=42, max_iter=1000)
    fa_null = FactorAnalysis(n_components=k, random_state=42, max_iter=1000)
    fa_act.fit(X_act_c)
    fa_null.fit(X_null_c)
    loglik_act.append(fa_act.score(X_act_c))   # mean log-likelihood
    loglik_null.append(fa_null.score(X_null_c))
    print(f"k={k:2d}  log-lik act={loglik_act[-1]:.2f}  null={loglik_null[-1]:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_RANGE, loglik_act,  'o-', label="IOI (activating)")
ax.plot(K_RANGE, loglik_null, 's-', label="non-IOI (null)")
ax.set_xlabel("n_components (k)", fontsize=12)
ax.set_ylabel("Mean log-likelihood", fontsize=12)
ax.set_title("Factor Analysis: log-likelihood vs k", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("figs/fa_loglik_vs_k.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Pick k based on elbow above — start with k=5 as default, adjust after seeing the plot
K_SELECTED = 5  # REVISIT after running the elbow plot

fa_act  = FactorAnalysis(n_components=K_SELECTED, random_state=42, max_iter=1000)
fa_null = FactorAnalysis(n_components=K_SELECTED, random_state=42, max_iter=1000)
fa_act.fit(X_act_c)
fa_null.fit(X_null_c)

# W_act, W_null: (144, k) — loading matrices
W_act  = fa_act.components_.T   # sklearn: components_ is (k, n_features) → transpose to (n_features, k)
W_null = fa_null.components_.T
print(f"W_act shape: {W_act.shape}  (n_heads=144 × k={K_SELECTED})")

# J_diff = W_act W_actᵀ − W_null W_nullᵀ — (144, 144)
J_act  = W_act  @ W_act.T
J_null = W_null @ W_null.T
J_diff = J_act - J_null
print(f"J_diff shape: {J_diff.shape}")
print(f"J_diff range: [{J_diff.min():.4f}, {J_diff.max():.4f}]")

In [ ]:
import matplotlib.transforms as mtrans

circuit_idx = [i for i, lh in enumerate(HL_FLAT) if lh in ALL_CIRCUIT_HEADS]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax = max(abs(J_act).max(), abs(J_null).max(), abs(J_diff).max())

for ax, mat, title in [
    (axes[0], J_act,  f"J_act (k={K_SELECTED})"),
    (axes[1], J_null, f"J_null (k={K_SELECTED})"),
    (axes[2], J_diff, "J_diff = J_act − J_null"),
]:
    im = ax.imshow(mat, cmap="PRGn_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for idx in circuit_idx:
        t = mtrans.blended_transform_factory(ax.transAxes, ax.transData)
        ax.add_artist(plt.Line2D([-0.03, 0], [idx, idx], transform=t,
                                 color="black", lw=1.5, clip_on=False))
        t = mtrans.blended_transform_factory(ax.transData, ax.transAxes)
        ax.add_artist(plt.Line2D([idx, idx], [-0.03, 0], transform=t,
                                 color="black", lw=1.5, clip_on=False))
    ax.set_xticks([l * 12 for l in range(12)])
    ax.set_xticklabels([f"L{l}" for l in range(12)], fontsize=9)
    ax.set_yticks([l * 12 for l in range(12)])
    ax.set_yticklabels([f"L{l}" for l in range(12)], fontsize=9)
    ax.set_title(title, fontsize=12)

fig.colorbar(im, ax=axes.tolist(), fraction=0.02, pad=0.04)
plt.suptitle(f"Factor analysis coupling matrices — log(Var_v), k={K_SELECTED}\n"
             "Black ticks = known IOI circuit heads", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("figs/fa_J_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## Thresholded J_diff: Circuit Graph

Threshold J_diff to get a sparse adjacency matrix — edges where |J_diff_ij| > threshold.  
Compare to known IOI circuit structure: are the strong edges between known circuit heads?

In [ ]:
# Empirical distribution of off-diagonal |J_diff| entries
mask_off_diag = ~np.eye(144, dtype=bool)
Jd_off = np.abs(J_diff[mask_off_diag])

# Threshold at mean + 2 std
thresh = Jd_off.mean() + 2 * Jd_off.std()
print(f"|J_diff| off-diag: mean={Jd_off.mean():.4f}, std={Jd_off.std():.4f}")
print(f"Threshold (mean+2σ): {thresh:.4f}")

# Find edges above threshold
rows, cols = np.where((np.abs(J_diff) > thresh) & mask_off_diag)
print(f"\nEdges above threshold: {len(rows)} (unique pairs: {len(rows)//2})")

print("\nTop edges by |J_diff|:")
edge_vals = [(abs(J_diff[r, c]), HL_FLAT[r], HL_FLAT[c]) for r, c in zip(rows, cols) if r < c]
edge_vals.sort(reverse=True)
for val, lh1, lh2 in edge_vals[:15]:
    r1 = head_role(*lh1)
    r2 = head_role(*lh2)
    flag = "** CIRCUIT-CIRCUIT **" if (lh1 in ALL_CIRCUIT_HEADS and lh2 in ALL_CIRCUIT_HEADS) else ""
    print(f"  L{lh1[0]}H{lh1[1]} ({r1[:10]}) ↔ L{lh2[0]}H{lh2[1]} ({r2[:10]})  |J|={val:.4f}  {flag}")

In [ ]:
# Summary: what fraction of strong edges are circuit-circuit, circuit-NC, NC-NC?
cc, cnc, ncnc = 0, 0, 0
for val, lh1, lh2 in edge_vals:
    in1 = lh1 in ALL_CIRCUIT_HEADS
    in2 = lh2 in ALL_CIRCUIT_HEADS
    if in1 and in2:   cc += 1
    elif in1 or in2:  cnc += 1
    else:             ncnc += 1

total = len(edge_vals)
print(f"Strong edges (|J_diff| > {thresh:.3f}):")
print(f"  Circuit ↔ Circuit:     {cc}/{total} = {cc/total:.2%}")
print(f"  Circuit ↔ Non-circuit: {cnc}/{total} = {cnc/total:.2%}")
print(f"  Non-circuit ↔ NC:      {ncnc}/{total} = {ncnc/total:.2%}")
print()

# Baseline: what fraction would we expect by chance?
n_circ = len(ALL_CIRCUIT_HEADS)  # 23
n_total_heads = 144
p_circ = n_circ / n_total_heads
p_cc_chance = p_circ ** 2
p_cnc_chance = 2 * p_circ * (1 - p_circ)
p_ncnc_chance = (1 - p_circ) ** 2
print(f"Expected by chance (uniform over pairs):")
print(f"  Circuit ↔ Circuit:     {p_cc_chance:.2%}")
print(f"  Circuit ↔ Non-circuit: {p_cnc_chance:.2%}")
print(f"  Non-circuit ↔ NC:      {p_ncnc_chance:.2%}")

## Loading Profiles: Which Heads Load on Which Factors?

For each factor (column of W_act), visualize the loading magnitude per head.  
Do factors align with functional groups (all Name Movers together, all S-Inhibition together, etc.)?

In [ ]:
ROLE_COLORS = {
    "Name Mover":          "#d62728",
    "Backup Name Mover":   "#ff7f0e",
    "Negative Name Mover": "#9467bd",
    "S-Inhibition":        "#2ca02c",
    "Induction":           "#17becf",
    "Duplicate Token":     "#bcbd22",
    "Previous Token":      "#e377c2",
    "Non-circuit":         "#aaaaaa",
}

colors_per_head = [ROLE_COLORS[head_role(l, h)] for l, h in HL_FLAT]

fig, axes = plt.subplots(K_SELECTED, 1, figsize=(18, 2.5 * K_SELECTED))
if K_SELECTED == 1:
    axes = [axes]

for k, ax in enumerate(axes):
    loadings = W_act[:, k]  # (144,)
    xs = np.arange(144)
    ax.bar(xs, loadings, color=colors_per_head, alpha=0.8, width=1.0)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_ylabel(f"Factor {k+1}", fontsize=11)
    ax.set_xticks([l * 12 + 6 for l in range(12)])
    ax.set_xticklabels([f"L{l}" for l in range(12)], fontsize=9)
    # Mark circuit heads
    for i, lh in enumerate(HL_FLAT):
        if lh in ALL_CIRCUIT_HEADS:
            ax.axvline(i, color="k", lw=0.3, alpha=0.3, zorder=0)

# Legend
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=r) for r, c in ROLE_COLORS.items()]
axes[0].legend(handles=legend_patches, fontsize=8, loc="upper right",
               ncol=4, framealpha=0.9)

plt.suptitle(f"W_act loading profiles — {K_SELECTED} factors\nColor = circuit role",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figs/fa_loading_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

## Scratch Space

Use cells below for ad hoc exploration. Delete before finalizing.